In [1]:
from bs4 import BeautifulSoup
import requests
import re
import json

In [2]:
def get_href(main_url):
    r = requests.get(main_url)
    url_list = []
    if r.status_code == 200:
        print(f'Request sent successfully to {main_url}')
        html = BeautifulSoup(r.text, 'html.parser')
        tmp = html.find_all('li', class_='col-xs-6 col-sm-4 col-md-3 col-lg-3')
        for i in tmp:
            book_url = f'https://books.toscrape.com/catalogue/{i.find("article", class_="product_pod").find("h3").find("a")["href"]}'
            url_list.append(book_url)
    
    else:
        print(f'Request sent failed to {main_url}')

    return url_list


In [3]:
def scrape_information(url):
    data = {}
    r = requests.get(url)
    if r.status_code == 200:
        print(f'Request sent successfully to {url}')
        html = BeautifulSoup(r.text, 'html.parser')

        data['url'] = url
        data['title'] = html.find('div', class_='col-sm-6 product_main').find('h1').text
        data['price'] = html.find('div', class_='col-sm-6 product_main').find('p', class_='price_color').text
        data['availability'] = html.find('div', class_='col-sm-6 product_main').find('p', class_='instock availability').text.strip()
        data['rating'] = re.search(r'star-rating\s+(\w+)', r.text).group(1)
        data['description'] = html.find('div', id='product_description').find_next_sibling('p').text
        tab = html.select_one('.table.table-striped').find_all('td')
            
        data['upc'] = tab[0].text
        data['product_type'] = tab[1].text
        data['tax'] = tab[4].text
        data['num_reviews'] = tab[-1].text
        
    else:
        print(f'Request sent failed to {url}')
    
    return data

In [4]:
def save_data(path, data):
    with open(path, 'a', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False)
        f.write('\n')

In [5]:
post_scraped=0
page=1

while post_scraped<20:
    main_url = f'https://books.toscrape.com/catalogue/page-{page}.html'
    url_list = get_href(main_url)
    
    for url in url_list:
        print(f'Scraping {url}')
        data = scrape_information(url)
        if data:
            print(f'Scrape successfully {url}')
            save_data('../data/books_to_scrape.jsonl', data)
            
            post_scraped+=1
            print(f'Post scraped {post_scraped}')
        else:
            print(f'Scrape fails {url}')
    page+=1

Request sent successfully to https://books.toscrape.com/catalogue/page-1.html
Scraping https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html
Request sent successfully to https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html
Scrape successfully https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html
Post scraped 1
Scraping https://books.toscrape.com/catalogue/tipping-the-velvet_999/index.html
Request sent successfully to https://books.toscrape.com/catalogue/tipping-the-velvet_999/index.html
Scrape successfully https://books.toscrape.com/catalogue/tipping-the-velvet_999/index.html
Post scraped 2
Scraping https://books.toscrape.com/catalogue/soumission_998/index.html
Request sent successfully to https://books.toscrape.com/catalogue/soumission_998/index.html
Scrape successfully https://books.toscrape.com/catalogue/soumission_998/index.html
Post scraped 3
Scraping https://books.toscrape.com/catalogue/sharp-objects_997/index.html
Requ